# RKS f1mo 分解 (SVWN, LDA)

DFT xc 贡献的 f1ao / f1mo 实现

对标 `06-3-decomp_f1mo_tpss0.ipynb`，但 SVWN 是纯 LDA，且 $c_K = 0$。

f1ao 组成（K 项消失）：
$$
\mathrm{f1ao} = \mathrm{h1ao} + \mathrm{J1ao} + \mathrm{vxc\_deriv1}
$$

LDA 下的 vxc_deriv1：
- **ipip 部分**：仅 LDA RHO，aow = wv[0]*ao[0]，与 ao[t+1] 缩并。
- **fxc 部分**：rho1 = $\sum$ ao[1:4]·ao_dm0[0]，与 fxc[0,0] 反馈。


In [1]:
from pyscf import gto, dft, lib
from pyscf.hessian import rks as rks_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from pyscf.grad import rks as rks_grad
from pyscf.dft import numint
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="SVWN").density_fit()
dat0 = np.load("nh3_r_svwn.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
nocc = mocc.shape[1]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
aoslices = mol.aoslice_by_atom()
ni = mf._numint

## 1. Reference f1ao from PySCF

In [5]:
mf_hess = mf.Hessian()
mf_hess.auxbasis_response = 2
f1ao_ref = mf_hess.make_h1(mo_coeff, mo_occ)
print("f1ao_ref shape:", f1ao_ref.shape)
print("f1ao_ref fp:   ", lib.fp(f1ao_ref))

f1ao_ref shape: (4, 3, 49, 49)
f1ao_ref fp:    -3.1969847300870637


## 2. f1ao 分解: hcore, J, Vxc_deriv1 (LDA: K coefficient is 0)

In [6]:
omega, alpha, hyb = ni.rsh_and_hybrid_coeff(mf.xc, spin=mol.spin)
print(f"omega={omega}, hyb={hyb}")

vxc_deriv1_ref = rks_hess._get_vxc_deriv1(mf_hess, mo_coeff, mo_occ, 4000)
print("vxc_deriv1_ref fp:", lib.fp(vxc_deriv1_ref))

gen_jk = list(df_rhf_hess._gen_jk(mf_hess, mo_coeff, mo_occ, with_k=True))
h1ao = np.array([r[1] for r in gen_jk])
j1ao = np.array([r[2] for r in gen_jk])
k1ao = np.array([r[3] for r in gen_jk])
print("h1ao fp:", lib.fp(h1ao))
print("j1ao fp:", lib.fp(j1ao))
print("k1ao fp:", lib.fp(k1ao))

omega=0.0, hyb=0.0
vxc_deriv1_ref fp: -4.579019717395777
h1ao fp: -34.59254245545264
j1ao fp: 35.97457744276134
k1ao fp: 1.2862567820116202


In [7]:
f1ao_recap = vxc_deriv1_ref + h1ao + j1ao - 0.5 * hyb * k1ao
print("f1ao decomposition verified:", np.allclose(f1ao_recap, f1ao_ref))
print("max abs diff:", np.max(np.abs(f1ao_recap - f1ao_ref)))

f1ao decomposition verified: True
max abs diff: 8.881784197001252e-16


## 3. 格点、AO、rho、vxc、fxc 准备

In [8]:
grids = dft.grid.Grids(mol)
grids.coords = dat0["grid_coords"]
grids.weights = weights = dat0["grid_weights"]
ngrids = len(weights)

# LDA: deriv=1 enough (need ao[0] and ao[1:4] for ipip and rho1)
ao = ni.eval_ao(mol, grids.coords, deriv=1)
print("ao shape:", ao.shape)

rho = ni.eval_rho2(mol, ao[0], mo_coeff, mo_occ, None, "LDA")
vxc, fxc = ni.eval_xc_eff(mf.xc, rho, 2, xctype="LDA")[1:3]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

ao shape: (4, 43328, 49)
vxc shape: (1, 43328) fxc shape: (1, 1, 43328)


## 4. AO 导数指标常量

In [9]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3

## 5. 自行实现 Vxc_deriv1

### 5.1 ipip 部分（LDA: 只有 RHO）

In [10]:
wv = weights * vxc[0]  # [ngrids]
aow = wv[:, None] * ao[0]   # [ngrids, nao]
vmat_ip = np.zeros((3, nao, nao))
for t in range(3):
    vmat_ip[t] += ao[t + 1].T @ aow

# Verify against gradient Vxc matrix
exc_ref, v_ip_ref = rks_grad.get_vxc(ni, mol, grids, mf.xc, dm0, max_memory=4000)
print("v_ip matches gradient Vxc matrix:", np.allclose(vmat_ip, -v_ip_ref, atol=1e-10))

v_ip matches gradient Vxc matrix: True


### 5.2 fxc 部分（LDA: 一阶密度只用 RHO 分量）

In [11]:
ao_dm0_0 = numint._dot_ao_dm(mol, ao[0], dm0, None, (0, mol.nbas), mol.ao_loc_nr())

wf = weights * fxc[0, 0]  # [ngrids]
vmat_deriv1 = np.zeros((natm, 3, nao, nao))

for A in range(natm):
    p0, p1 = aoslices[A][2:]
    # rho1[t, g] = sum_u ao[t+1, g, u] * ao_dm0_0[g, u]   (no *2 factor)
    rho1 = np.einsum("xpi,pi->xp", ao[1:, :, p0:p1], ao_dm0_0[:, p0:p1])  # [3, ngrids]
    wv_f = wf * rho1  # [3, ngrids]

    for t in range(3):
        aow = wv_f[t][:, None] * ao[0]
        vmat_deriv1[A, t] += aow.T @ ao[0]

    vmat_deriv1[A, :, p0:p1, :] += vmat_ip[:, p0:p1, :]
    vmat_deriv1[A] = -vmat_deriv1[A] - vmat_deriv1[A].transpose(0, 2, 1)

In [12]:
print("vmat_deriv1 (my) fp:", lib.fp(vmat_deriv1))
print("vxc_deriv1_ref fp:", lib.fp(vxc_deriv1_ref))
print("max abs diff:", np.max(np.abs(vmat_deriv1 - vxc_deriv1_ref)))
print("allclose:", np.allclose(vmat_deriv1, vxc_deriv1_ref, atol=1e-8))

vmat_deriv1 (my) fp: -4.579019717395777
vxc_deriv1_ref fp: -4.579019717395777
max abs diff: 2.220446049250313e-15
allclose: True


## 6. 组装 f1ao 并变换到 f1mo

In [13]:
f1ao_my = h1ao + j1ao - 0.5 * hyb * k1ao + vmat_deriv1
print("f1ao_my vs ref:", np.allclose(f1ao_my, f1ao_ref))
print("max abs diff:", np.max(np.abs(f1ao_my - f1ao_ref)))

f1ao_my vs ref: True
max abs diff: 2.220446049250313e-15


In [14]:
f1mo = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_my, mocc)
print("f1mo shape:", f1mo.shape)
print("f1mo fp:  ", lib.fp(f1mo))

f1mo shape: (4, 3, 49, 5)
f1mo fp:   4.596981078717835


In [15]:
vmat_deriv1_mo = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, vmat_deriv1, mocc)
print("vmat_deriv1_mo fp:", lib.fp(vmat_deriv1_mo))

vmat_deriv1_mo fp: 0.4744765644859929


## 7. 验证：与 PySCF 直接所得 f1mo 比较

In [16]:
f1mo_ref = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_ref, mocc)
print("f1mo_ref fp:", lib.fp(f1mo_ref))
print("f1mo matches ref:", np.allclose(f1mo, f1mo_ref))
print("max abs diff:", np.max(np.abs(f1mo - f1mo_ref)))

f1mo_ref fp: 4.5969810787178425
f1mo matches ref: True
max abs diff: 2.6645352591003757e-15


In [17]:
dat = dict(np.load("nh3_r_svwn_decomp.npz"))
dat.update({
    "vmat_ip": vmat_ip,
    "vxc_deriv1": vxc_deriv1_ref,
    "vmat_deriv1": vmat_deriv1,
    "vmat_deriv1_mo": vmat_deriv1_mo,
})
np.savez("nh3_r_svwn_decomp.npz", **dat)